# Phase 2b Stage 2: Entity聚类

对Stage 1.5的626个entities聚类，合并同义词

- 方法: BERTopic（自适应聚类数）
- Canonical name: 按字母顺序选择

In [ ]:
import sys
sys.path.insert(0, '..')

from src.clustering.entity_clusterer import EntityClusterer

## 1. 加载数据

In [ ]:
# 初始化
clusterer = EntityClusterer(
    stage1_5_result_path='../results/entity_redistribution_stage1_5_redistributed.json'
)

# 加载Stage 1.5数据
stage1_5_data = clusterer.load_stage1_5_results()

## 2. 测试单个relation（visual_theme）

In [ ]:
TEST_RELATION = 'visual_theme'

print(f"测试relation: {TEST_RELATION}")
print(f"Entities: {len(clusterer.entities_by_relation[TEST_RELATION])}")

In [ ]:
# 生成embeddings
embeddings, entities = clusterer.embed_entities_bge(TEST_RELATION)

In [ ]:
# BERTopic聚类
topic_model, topics, probs = clusterer.cluster_with_bertopic(
    relation=TEST_RELATION,
    embeddings=embeddings,
    entities=entities,
    min_cluster_size=2,
    verbose=True
)

In [ ]:
# 查看每个topic的详细entities
print(f"\nDetailed topic breakdown for {TEST_RELATION}:")
print("="*80)

for topic_id in sorted(set(topics)):
    topic_entities = [entities[i] for i, t in enumerate(topics) if t == topic_id]
    topic_entities.sort()  # 按字母顺序排序
    
    if topic_id == -1:
        print(f"\nTopic {topic_id} (Noise): {len(topic_entities)} entities")
    else:
        print(f"\nTopic {topic_id}: {len(topic_entities)} entities")
    
    for entity in topic_entities:
        print(f"    {entity}")

## 2.1 参数调优实验

调整BERTopic参数以改善聚类质量（避免大的"垃圾桶"cluster）

**关键参数**:
- `min_cluster_size`: HDBSCAN最小cluster大小（越大越严格）
- `n_neighbors`: UMAP邻居数（影响局部vs全局结构）
- `n_components`: UMAP降维维度
- `min_dist`: UMAP最小距离（控制点的紧密程度）

In [ ]:
# 参数实验 - 修改这些值来测试不同的聚类效果
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# 🔧 调整这些参数
PARAMS = {
    'min_cluster_size': 3,      # 试试: 2, 3, 4, 5
    'n_neighbors': 10,           # 试试: 5, 10, 15, 20
    'n_components': 5,           # 试试: 3, 5, 7, 10
    'min_dist': 0.0,             # 试试: 0.0, 0.05, 0.1, 0.2
}

print(f"测试参数配置:")
for k, v in PARAMS.items():
    print(f"  {k}: {v}")
print()

# UMAP配置
umap_model = UMAP(
    n_components=PARAMS['n_components'],
    n_neighbors=min(PARAMS['n_neighbors'], len(entities) - 1),
    min_dist=PARAMS['min_dist'],
    metric='cosine',
    random_state=42
)

# HDBSCAN配置
hdbscan_model = HDBSCAN(
    min_cluster_size=PARAMS['min_cluster_size'],
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# BERTopic聚类
topic_model_tuned = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=False,
    calculate_probabilities=True
)

topics_tuned, probs_tuned = topic_model_tuned.fit_transform(entities, embeddings)

# 统计结果
n_clusters = len(set(topics_tuned)) - (1 if -1 in topics_tuned else 0)
n_noise = sum(1 for t in topics_tuned if t == -1)

print(f"聚类结果:")
print(f"  Clusters: {n_clusters}")
print(f"  Noise: {n_noise}")
print()

# 显示每个topic
print("Topic分布:")
for topic_id in sorted(set(topics_tuned)):
    topic_entities = [entities[i] for i, t in enumerate(topics_tuned) if t == topic_id]
    
    if topic_id == -1:
        print(f"  Topic {topic_id:3d} (Noise): {len(topic_entities):2d} entities")
    else:
        print(f"  Topic {topic_id:3d}: {len(topic_entities):2d} entities - {', '.join(sorted(topic_entities)[:3])}")

# 检查最大的cluster
max_cluster_size = 0
max_cluster_id = -2
for topic_id in set(topics_tuned):
    if topic_id == -1:
        continue
    size = sum(1 for t in topics_tuned if t == topic_id)
    if size > max_cluster_size:
        max_cluster_size = size
        max_cluster_id = topic_id

print(f"\n最大cluster: Topic {max_cluster_id} 有 {max_cluster_size} 个entities ({max_cluster_size/len(entities)*100:.1f}%)")
if max_cluster_size > len(entities) * 0.4:
    print("⚠️  警告: 最大cluster过大，可能是垃圾桶cluster")

In [ ]:
# 查看调优后的详细topic breakdown
print(f"\n详细topic分组（调优后）:")
print("="*80)

for topic_id in sorted(set(topics_tuned)):
    topic_entities = [entities[i] for i, t in enumerate(topics_tuned) if t == topic_id]
    topic_entities.sort()
    
    if topic_id == -1:
        print(f"\nTopic {topic_id} (Noise): {len(topic_entities)} entities")
    else:
        print(f"\nTopic {topic_id}: {len(topic_entities)} entities")
    
    for entity in topic_entities:
        print(f"    {entity}")

In [ ]:
# 生成entity mapping
entity_mapping_test = clusterer.generate_entity_mapping(
    relation=TEST_RELATION,
    topics=topics,
    entities=entities
)

In [ ]:
# 查看合并的groups
print("\nMerged groups (showing only groups with >1 entity):")
print("="*80)

canonical_groups = {}
for orig, canonical in entity_mapping_test.items():
    if canonical not in canonical_groups:
        canonical_groups[canonical] = []
    canonical_groups[canonical].append(orig)

for canonical in sorted(canonical_groups.keys()):
    group = canonical_groups[canonical]
    if len(group) > 1:
        print(f"\n{canonical}:")
        for ent in sorted(group):
            if ent == canonical:
                print(f"  ★ {ent} [CANONICAL]")
            else:
                print(f"    {ent}")

## 3. 处理所有relations

In [ ]:
# 处理所有relations
all_entity_mappings = {}

for idx, relation in enumerate(sorted(clusterer.entities_by_relation.keys()), 1):
    print(f"\n{'='*80}")
    print(f"[{idx}/15] Processing: {relation}")
    print(f"{'='*80}")
    
    n_entities = len(clusterer.entities_by_relation[relation])
    
    # 太少的entity跳过聚类
    if n_entities <= 2:
        print(f"⚠️  Skipping (only {n_entities} entities)")
        all_entity_mappings[relation] = {
            ent: ent for ent in clusterer.entities_by_relation[relation]
        }
        continue
    
    # Embeddings
    embeddings, entities = clusterer.embed_entities_bge(relation)
    
    # BERTopic聚类
    topic_model, topics, probs = clusterer.cluster_with_bertopic(
        relation=relation,
        embeddings=embeddings,
        entities=entities,
        min_cluster_size=2,
        verbose=False
    )
    
    # 生成mapping
    entity_mapping = clusterer.generate_entity_mapping(relation, topics, entities)
    all_entity_mappings[relation] = entity_mapping

## 4. 统计

In [ ]:
print(f"\n{'='*80}")
print("Final Statistics")
print(f"{'='*80}")

for relation in sorted(all_entity_mappings.keys()):
    mapping = all_entity_mappings[relation]
    n_before = len(mapping)
    n_after = len(set(mapping.values()))
    compression = n_after / n_before if n_before > 0 else 1.0
    
    print(f"  {relation:25s}: {n_before:3d} → {n_after:3d} ({compression:.1%})")

total_before = sum(len(m) for m in all_entity_mappings.values())
total_after = sum(len(set(m.values())) for m in all_entity_mappings.values())

print(f"\n  {'TOTAL':25s}: {total_before:3d} → {total_after:3d} ({total_after/total_before:.1%})")

## 5. 保存结果

In [ ]:
# 保存
clusterer.save_results(
    all_entity_mappings=all_entity_mappings,
    output_path='../results/entity_mapping_bertopic.json',
    metadata={
        'embedding_model': 'BAAI/bge-base-en-v1.5',
        'min_cluster_size': 2,
        'canonical_selection': 'alphabetical'
    }
)

print("\n✅ Phase 2b Stage 2 完成！")